# Stage 2: Acoustic Source Localization & Spatial Beamforming
**ECE Minor Project**: *Spatial Multimodal Edge-AI for Rotating Machinery*

This notebook implements and benchmarks:
1. **Stage 2A**: Known 4-microphone ULA geometry (spacing = 20.32 mm, aperture = 60.96 mm)
2. **Stage 2B**: Pairwise super-resolution GCC-PHAT TDOA estimation vs. analytical ground truth
3. **Stage 2C**: 2D Steered Response Power (SRP-PHAT) acoustic camera heatmap backprojection
4. **Stage 2D**: 25-point off-grid benchmark suite & SNR robustness sweeps (Clean, 20dB, 10dB, 0dB, Multipath)

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, os.path.abspath("../src"))
from stage2_acoustic_localization import ULA_4MIC, simulate_array_signals, gcc_phat, SRPLocalizer, FS, SPEED_OF_SOUND
print(f"[+] Loaded Array: {ULA_4MIC.name} ({ULA_4MIC.num_mics} mics, {ULA_4MIC.num_pairs} pairs)")

## 1. Stage 2B: Pairwise GCC-PHAT TDOA Verification
Evaluate sub-microsecond delay estimation against analytical ground truth.

In [ ]:
test_src = np.array([0.75, 1.25]) # 75cm right, 1.25m front
rx_sig, true_tdoa = simulate_array_signals(test_src, ULA_4MIC, FS, snr_db=25.0)

print("Pairwise TDOA Ground Truth vs GCC-PHAT:")
for p_idx, (i, j) in enumerate(ULA_4MIC.pairs):
    d_ij = np.linalg.norm(ULA_4MIC.mic_coords[j] - ULA_4MIC.mic_coords[i])
    est_tau, prom, _, _ = gcc_phat(rx_sig[:, i], rx_sig[:, j], FS, d_ij)
    err_us = (est_tau - true_tdoa[p_idx]) * 1e6
    print(f"  Pair M{i+1}-M{j+1} (d={d_ij*1000:4.1f}mm): True={true_tdoa[p_idx]*1e6:+6.1f} us | Est={est_tau*1e6:+6.1f} us | Err={err_us:+5.2f} us")

## 2. Stage 2C: 2D SRP-PHAT Acoustic Camera Heatmap
Backproject spatial steered response power onto a 2D floor grid.

In [ ]:
grid_x = np.linspace(-1.5, 1.5, 121)
grid_y = np.linspace(0.4, 2.5, 85)
localizer = SRPLocalizer(ULA_4MIC, grid_x, grid_y)

est_pos, heatmap, est_bearing, conf = localizer.locate_2d(rx_sig, FS)
true_bearing = float(np.degrees(np.arctan2(test_src[0], test_src[1])))

plt.figure(figsize=(9, 6), facecolor="#0a0e17")
ax = plt.gca()
ax.set_facecolor("#101726")
im = ax.imshow(heatmap, extent=[-1.5, 1.5, 0.4, 2.5], origin="lower", cmap="inferno", aspect="auto")
ax.plot(test_src[0], test_src[1], "wo", markersize=9, markeredgecolor="#00ffff", markeredgewidth=2, label=f"True Source ({true_bearing:+.1f} deg)")
ax.plot(est_pos[0], est_pos[1], "rx", markersize=10, markeredgewidth=2.5, label=f"Estimated Peak ({est_bearing:+.1f} deg)")
ax.plot(ULA_4MIC.mic_coords[:, 0], ULA_4MIC.mic_coords[:, 1], "c^", markersize=7, label="4-Mic ULA Array")
ax.set_title(f"2D Acoustic Camera Heatmap | Bearing Error = {abs(est_bearing - true_bearing):.2f} deg", color="#ffffff")
ax.set_xlabel("X (meters)", color="#b8c2d1")
ax.set_ylabel("Y (meters)", color="#b8c2d1")
ax.tick_params(colors="#b8c2d1")
ax.legend(facecolor="#0a0e17", edgecolor="#2a3b5c", fontsize=9, loc="upper right")
cb = plt.colorbar(im, ax=ax)
cb.set_label("SRP Power", color="#b8c2d1")
cb.ax.tick_params(colors="#b8c2d1")
plt.show()
print(f"Localized Angle: {est_bearing:+.2f} deg (True: {true_bearing:+.2f} deg) | Error: {abs(est_bearing - true_bearing):.2f} deg")